# CinderTrace Atlas — direct catalogue pilot

**Goal:** one reproducible observed-data extract for Sri Lanka, 2023, directly from `portolan-mirrors/firms-catalog`. This is a fresh Atlas run, separate from the earlier HDF/swath notebooks and existing application datasets.

Use a fresh CPU Colab runtime, from VS Code or browser Colab. No Earthdata/LAADS credentials are required for these public mirror files. Google Drive is used to save your run. No browser upload/download widgets are used.

This produces observed detection counts and a small handoff bundle. It does not create harmonized activity, infer fire events, certify physical absence of fires, or write to the existing app.

Sources: https://github.com/portolan-mirrors/firms-catalog and https://github.com/portolan-mirrors/firms-catalog/blob/main/catalog/detections/README.md


## 1. Install the query tools

The mirror uses GeoParquet 2.0. Use DuckDB 1.5.x and the spatial extension, rather than the old notebooks' DuckDB 1.3.2 environment. Resolved versions are saved to the run. Restart the runtime if Colab requests it.


In [12]:
%pip -q install "duckdb>=1.5,<1.6" pandas pyarrow requests shapely


## 2. Start a fresh Atlas workspace


In [13]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,uuid,importlib.metadata as metadata,subprocess,sys
import requests,pandas as pd,duckdb
from shapely.geometry import shape

drive.mount('/content/drive')
assert tuple(map(int,duckdb.__version__.split('.')[:2]))==(1,5), 'Use a fresh runtime with DuckDB 1.5.x.'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
RUN=Path('/content/drive/MyDrive/cindertrace_atlas/runs')/RUN_ID
RUN.mkdir(parents=True,exist_ok=False)
WORK=Path('/content/cindertrace_atlas')/RUN_ID;WORK.mkdir(parents=True,exist_ok=False)
YEAR=2023
BBOX=(79.4,5.8,82.0,10.0)
SOURCE=f'https://data.source.coop/portolan-mirrors/firms-catalog/detections/year={YEAR}/detections.parquet'
versions={p:metadata.version(p) for p in ['duckdb','pandas','pyarrow','requests','shapely']}
(RUN/'environment.json').write_text(json.dumps({'python':sys.version,'packages':versions},indent=2))
(RUN/'requirements.lock.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
print('RUN_ID:',RUN_ID)
print('Source:',SOURCE)
print('Versions:',versions)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_ID: 20261114T183601Z_07884c3d
Source: https://data.source.coop/portolan-mirrors/firms-catalog/detections/year=2023/detections.parquet
Versions: {'duckdb': '1.5.6', 'pandas': '2.2.3', 'pyarrow': '23.0.1', 'requests': '2.32.4', 'shapely': '2.2.0'}


## 3. Load the frozen study boundary

This embedded reference preserves the existing Sri Lanka study outline. It is geometry only; no old fire totals or models are imported. Polygon/pixel-centre support is explicit, and independent boundary review remains part of the spatial audit.


In [15]:
import base64,zlib
BOUNDARY_SHA256='838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'
BOUNDARY_COMPRESSED="""
eNqFfcvOBrlx3asIWhsDFlksktkGyCoBAmQZeCHYiiFElgxFXhiG3j3nkM1ikb9ljRaa+Zsfm5e6nLr2v//6z//2L7/99X/51a//
229/8+d//dNv/+sff//73/7Dn3/3xz/8+u9+9ev/s/74/zDgf//7O5QD/uVPf/yX3/7pz7+bQ/7917/7R/xfwt//8Jt/nkP/159+
96v//ps//N/f/Pov+Os//faP//zbP//p3+bYPdv/+Nff//l3//OPv/+3f1rv/Ic//vFP//i7P/zmz+u1+KeNX1pTk/p3v+q/ZBvV
1P7+7361/l6yio35T5vP20it+fNeNY+6ntv6fetWR5r/qA8rxcY1jeVRhj//8fq/55O5tC6io8o3IceU0npu+Xvp+OboSXOvZQ3A
7P17nfU9QHK3XtZfxxpXUhNbU5fsE5n1rmtYn+OyahryLb/4fKpS1gtzEsPJrJ/4C1MfOaU4kTTs0x+nkUTC2RVN1vQ7u+Lj/uYB
nMNqvWFNNbyy4v24pfVraX7go+c+B+gopfoyfEDPaj2FG9Oeaqv7L93HWZJqYXGKE7W0X7gPq+H4pEdCUZuXs8bl6vMNHEqN81kx
cXobnLqsx5NgFAQodS9fzh2WvHc9r0i1j16+M/NdjlGzlXBcitlb+S6/hdf23mQNADHkvgbU4QNyyuuY6igtt2/7hwFa7XuVa5xZ
S/vg8uG3UXHy8Zh6a1n9ePAfzda9SscU38hzG1VTPD2+s0v5cbu1lZHnAN5+/04057Pev0FHgeYgEUBAfifjF8ENlr7peF+JZcz4
HcEcllqu1dluZBOw1LeQ+VxyhiT51m5hnGl3NsY48Cao+GFPkFt2dm9rvkTJ8Bz5kALhM6732tC+r643Utz8OyjD9KVsyJlmXeLv
tfDEX7FiueDG4rq1peLiC4fdi86/43JchtQislZvOOOxz8dXZxBhpfiNc1Yt2fbVNR+HK92vBclLfjkGEgEHdB2XZYi9fUwgZq2T
9Ej3Wfu5j5p6JG2Illpc7l3jsgaWHK2Civ3+BcsqLc7Tqjbn1T4g1GsLrAF6Hr6MPsCZOQfSx/gs4+NEvwUcWK81CJCBZcmWf83P
dSSrl/wbtdax6VPPrQqOSdaxKJn15aSh0KDNOYnnj41oezkf+jBri+QBKnABYh26QWqcpnXN7aUHalkdEukBtCnWHqViHdycF7li
VxA730TFJ4LAK5vYF1eT6vU+zwZS11rWc0iCstm1HIn2n0uHI0hAtKDmoyPGLxUCcmwx5KRqDYKhrOeldvM/D9xsDVuHLMzZztFk
HFlkdPAWFG5/dWMuTfWaB/Ic4vAb18/WTbIEjtEBVdS2MpQzHycIV6fN8OotiFyHJK1dcnivQmk4XIBkB/mvbSfCgU0Mvj+wYd2q
c+1PTbV92qQ2H/e3jjmCL/yvxz1CflLrvGwg5J0oBC1PDfyCNAClRU2EDTz/CydCAEKaXlsYQBf7MvyoerGeZa19AGT53VFBjS1k
+3pecenfnYhPoJCqvf31hagM9Rtaw6oV/cH+FQNru8Yp+OBMQywZjw9iRQ+EhBKGEow/Fyq3/qhOJaC1wItQWAqqCqrFFhNifigk
h44AHdQx4arxwrGlsJMypBik4nzewKvhvan0eArVIJM3WtCDtsAK2eK4Rtn9A4EDudVy3w5+0fbh6wHOwCzFVQXGQfoDCH5/8XFg
3yz3fFCxZs9lQoGPrY/Gd+cFMO1Mo+m+Jag3iL/vFz4M8CX1sCrsueuWOI5R/zbPBOwEcgcwXoMgO7Z+LweoVyVWO1sESoJ1kh7g
B11h7TqJrqSlV7aBhvs+Ucj3/kj9Rmxl1+tAUOI04Hj2r687GiO5SbuYozdA562LAmoFTMiBSs1gagw5KgSqNccDrVXLJhtHWI20
W+s1LuGo+gP6yWD1IoeKU9EfcrtBMsh1jw0KcRPK0XB/c5tRnirpN4xtQh7QH5QNY0kiBxiQoOvRwHkNCq5EyTAmUj826t9437e2
nn7JsI/XnfdSITjm9fDvtTRrJZxXh0YDpolChONA9FrnInB5OPeNvds3AOZ2HvlbJKzETVR7Ath5CsW7CAC6MH1n3+s3gIK9Lc2Y
KUctQvH5vCRLkQDAJ1CW/rhC1rRwWAqZUx09f8OgqmEUxmEQWdryfgxyb2ZBTfMw+oZutnfbIJnqQligGhkOwc6AQhMu3DGOHZJn
RHnCZQPumEZkDuUFPToi785xkHK2nkMpmkTpy/fhclK7Fg5NlPTSjRwHfQpui+sSQNi0kck+TohNDI2QFBD2Q4Z8rBo2Mx8DHOL9
zzkBI0ApRdElsKfqfl3x+VoHPIjLgopooKPvORREkXGUFe2QUY4g3vOAIACr1nMI+bEdPf6c/oyql0WF5Wy6yZtM6NuBAI32LI7T
0nkOcJi33YibCRJ7bQf0Ebe9lOg+z+HjDIzSol0Ghqwuuc2XDe2q3Y+b9gjgxZ7vHJPwzjUI28R1jv3Lfbsgglyz3eYz6HBbMj5M
C+zJZYZDtWwz3HxdQl9a9KYNTl3kPg7AEpit6qfJYVCVLjsGgBSsqGUWAtJsZ1s76wV562dXfgCNf4ZEAJJbf4Z5uzWZnA2At/Jn
TxbQ51k4LCyJviHMC61ZH6mFmxW1z5TsVIb+ADvdwHKtoAIN1k3YZ4IK0XlZrrjHEm5MuLD4nDhkn3T1eUCNLegMjMMkwHC+Uxxa
a98DkNA4R9epb8NOM70e+giiUYCUs8aF3PMUyLyNmvY8/dwQDHTsPj4GHGzJ168QztsKWTcNKi/bwjrDwHY1uvZwv3VsKe5ibIBL
co3eVxwb9JI8Yhi3Drm0BsDCsu0j7D4APwJmiKQJ+y1viezaaRAMDosL65jatkAgU4DVrnsWGP31okgBH2V6S697rMAYWyOJj2vg
vH2fNS/bng9wFH1SNGXUcmnwzzAnR62BHgFTeJuXnsA4mMTRjsS4Splx+TQ5rqcssgbgJ9DIEcZzAPin7gEZJnuNKGfOUPvyZnao
bXdmlP2Y571dOvMg8BJgkxY9fRgHZVirWNhYG1yuXRIb42jOLK/nBDT9cnrPATAsU+Q0jGtlW9B+kngXAVpwDkG30s/yzbcvAooM
1z+f02W/59kHjWVLT5FgaF2V2/vMcRlQ5/KMAbBmpxccbG4XodPF1bcb+GNPjjOo63k+Bv6WLW337sFogH0WHGxQPMBs7aFPjrO1
XFx/Kf532PtyucywiDI2uYLUtNYYAajNcjNfnhII9EB2IHpArRJt//kamMe5xte06YbxeSBjJcfnmNbhffFTVQUBXOMUdrafRqZb
NRIf5MkHIuZeE6TT+hW9NP3HYUsvcu0G2Da5A8HnKXXpjmrASztu5q/JtPJ7EImQ0zDW/efgblskXxdNRH/LtwxpERXQowTD5tzl
SFeICnajuNrzw8Dxi3vi1+swr8+Cm3UJ+V1thV64rM85DvOsx2PLRXAJxGu1a42N/mm5NDzH4dSHXYcB+bIRbTnjoDHzFQSxpbqf
PWExek0H2S/bxMpnOuys2TWdwc7UV3gAokwAfAgGbKZb4dUwrru/eR35vDuL/vK5Plj6OQe1A6MXl7cvS858EOPXfDSW3cFcwh1m
ly1rHE7JAd3ZLxC5rYnoJBlHWuG0ovYHQ2DdZ72QKLbuD6/J2/7oJQz4Yo44FBz2eHlGYGWslRVino3l21+Vl9QZuTs3wIBNGgEX
DiunQ2c4Jo04iw5Tvxb8a/+ok9JjO7lK9gEg+W0W1W+b5DgX9oPBy/UA6s6dlv7+BjJf+AvAxDZwozppEDQlygrIuLHNX/MVgOby
2jdNuVJeCsRfVGIkpsoUyO9CcNBAPHEcNbFekbE5Duq5SAy20qrclsyhGFhBMko4dzpY6jbEzxGDj8p38C2f1wAmLABOb0tp/SfD
pI8eoed7YEwgwRYvhDBWD7nRnI6nCvLN2urLF5numytUbNyiP2c0T/WaB6Tkis7HAQv3XGKIF/IptfMY3PzFmFUOxx79yODies4A
0dFnJLL6Lay0Lc01H0VmkCnfLyWsCHZX1Qg4Fz7tl19mTgB2G1fMHYe7HbpH8QPC5KFXEBvWjBudrlnpvdV20QNEZpVH9wGibGqg
c/37M5QuGPDi8jSAijZf72G0Wb8QNxZRy40dC0yJ0jezEfruBwz4twimQNR0QD6nythbuw6lNLop9+ohGqEtopYpjFVsLeNSD5xc
V2oLjuQ4jlyFMxL25ZAA62iNZjHfUk9wYg0TRkHLA2YYzqgasT00B7B1fRCp4s8H+M1xgPtt+wOdeaAZIABbIHqa8UeWApZAfsUr
hnSja9GfQ5+19UNcbhvhh5JS5BJItTTsAT4gaZxGNL6hO7CbElzyHAbyrpfTQ6SIU66LCoUKt8VDDKIX94j6+5hWskAtnuN82+U2
4ZsAvC5wS7lazvOWmfY0H+D9/udBhLzGCyGhXe4RIW3DVsPKGu3DBprbDwTqJkSg5nOYIu1RS6SOOiSO6wy8yaNcK080HX8T56Ml
Le98UOmpHLrmfL24rF/pJQfqNXpFijlvn3Ewd0KiTqPd3rqDNKhICDWLryH42jfnhFgz42HHDcVxDBLWR/dVSBx3sa73zeNuDztW
YgRZCxfmc5RHjANQgl+POuA4bS4D8czG93d6yPpZJ5j7UGKb+Q5V/TkINUuO0+apW799qB8/cIbmeE3Yb7niPXMYtlrXdYPXyuN9
FmLZCSI2epzrGY6O3AQHWeP29gDYa5tC/E2dhpDGC8hHSuIwMhD6OqYeBBLeb017IMsOxOHBcfHjNoZQj9OnMd/m5DWI0wtUStJI
vp0mt8c2N8PRjzAkxfmok/d6DdfPdLpDvoTYvlmovd43MlpvgTJTT4TytzDwZ2tbEKfuixd/Dy/b4kTQ2+OotX38RpGfSiAb2FHa
3au+h/VGJB64pTEgvNWAKxMoKCxyzQON6OlGbg82AoQa2QSKGRdVHlDU6HJqI1wfZHirbt7sa8HhwqwtgTqAYoCy7NbGECBMnQrU
DyS+Xe54bLW74TUfM+NS04jRrjkOjHyNy2DW7N5s9XHghy7h9HOlJ+fxL3XiszKJRJhD6M6Jsy4o5+2ZnacFsEOP8gOUO+7WUnwf
NEu2oQ+S5T4LqM4gUysQjT1U0yGupBzmx7hGD8B7iyB9aB6ZA4CNxeSdiLHy4Iq0XxjNrjs64uQHTZFnGBcDKt0/+miDXhho6M4v
xpSG4ml8bqx2Ztq0o60wrkAvbFTmHgBym2qL4yCF1X1pftMT9A8OaJ2v1McW7JPGJOwQuHi4OvP1g+NOvsIcBuka6BczaohtGj11
UKe+L/JEXj9MTMzMj1qlfyqvO4X2tmqvcQRUxCmcQ4z5N1q3E9NFCuATDyzcvekkv9trQY/iKBouxBKdp+O5OMiHPtqJvtkvgP7Q
yf6cur/NG4DNBOq/Mlo5AKh5/Q4MfVw9eDm09DwQqGV6rPYDE0j8Gl6o1GPOmy7KYJTYes7EBleQBq0ibU1cIP88TltcNoNwTMNB
KqBVci/BEeHJgpWPYULgWB4FSl1Z+bjMHGC7jxmGurauYRbYA2B9ienPn5rzLIpJhmU6NctjOtFlNM0VDmBGvf2AHRDder2wEmC0
h4GgLKqWGl+oTHtzvMmQbguHBCw4DqxnQKKP+JZMv1C5bSulQR5SWjEMhqyjQpd7CoPXgufZfsGF02x6tAlojaHROK4yyd4eooWK
s3FJG5qmfcsV1wPAbdlzGOb6GFsY2yXnUqlwMWPEcRgoO6Tg4KIUUuWkBsC4auX1OUP8zBy+Q9x0guc7N3lOBKWm67lmp+3CPOf1
M0gj2ftxKQQJA+Eyb5POUFf/brTSUSl1rAGVqdnPi7E2KIQ1Q4ex6xLieCfqwr1GR9VKIAxhzBkdAGNH3QO2ES9f8DfhOihJAgFC
doDkPRgDQFeXYBElAE7PpQDHldoi0UDlMS72vgik1eedAGbDyDheLib5RA5g/FqON4rsVy0QEV1ijjGPWylZzpckFSAhPw9aVRYq
X/gahvfzG+QarHc4iGqOg/Lrj4XOJNwiayLm24/ze8La9XcKgHaFH/lDZcA/7hcgtrmN6/PQeuxxvcK0wX7LvcSTSJG1ZVap1DcI
CDL0+MX31gIx5LG5OjFpUJwyk/H8WBIzStd5JHohPQrNOOYlCxJjzRKrBWaUl5EnC+tMPFT33nlwFqxrwcvHcaCv57bXuD5vOTUa
5Se9gtuyIBFTZWQjPblLkFXmMbi5LVwLttgu0cSsF1BqSHfguKqeg+bZBR386jUCUzQkKIbihRjmyTidWiLOB/7x9GSPdoO5tYSg
rRGCtn1fadbV9DTpAlqpDE/2gajP9+9wjcVxjnjyENAyfl6ZVgJxZiepKPVQnMTnifk8F9LEOBKdWhwHS+skmPt8IFmVcY2DSPGc
KdCABr9TJUJO4kEMT9KCxOxjPWctx/l72T/rPKAnN4gHlpgcVJlOVN1RZef3zOg4x1WZHUCT6/KzMYesJ+sjrtNoq7xJYuCDOlPe
5kT5mKbZB7RyFOCaCNrIf9+hF8yu9TALb4OifX3Uby3kKXMcAI5PA/iudmKqeAzFO1x47NwLop1YDshx0Nl1ywlPRcusqjrmL8fB
FtjFAZ7cZMJESXPhWcmnTCN8UluYKgiUHU8BOtXz/ioTNzTHaUCAzrss2xhpLRekJbtyrm3irEyUCtbLHActsu3a4eMYjl0DqGW2
GPN1gkXAEBqvvTCddlM3NeMIkRKeC9bm61DoI2px/h0421PtIB7NbM3Hjd72XmINH0y574eJsPjhBwXGrzebMlzhVY57gYDksDry
RwE2vGZz+IDSNdShzB3I4SBlXu5I13Or4tVQzedhzKXclNQ9ji5y3jePKIxjXYGnye0TUmLHj5EAsPt48+PolIUQWwNS8SRnf46n
ZRxcMznE3IXpeXGQ+KYhN50LolQZD8cV0txaDx2NujeejWbPCelPgVqbS4ZNCwzetBBOomBm9k715wWG+NIcqRIrPbyamdOsn8Sn
EClPxhwmrz1HhJOYt+75YhkkoyH8R5UCu9yd4Nnn6dMGDSoEtGg7EezkEedBmBogBVVZ38TiJAYJCwRVL5WnjDo+JMSMIkvjmg8L
qb4uoWMhQhjKEBX/fRoMHqwfQkbs3wnvy1I8F+ZZ7ac4R7Dg/LNNzLwfZOZMrwuZLCXPjTLpETy1sNKU9BdETPxZzyFBjMhYmOJx
meYcxzjcAl2w6OpGn3nfbOq4jutmWZI7PKPCX8iKOlsocNAgK5d5ywGF5v+yXHgpt/sDA6pA1l+GEe/ANfQexlQabdEAZCpBk/d9
pTX7DDFWa20+6T4gAwWOZZEyt9fO3zUlCWCeAcO8+ccVHauP1n7KDIfIk4vJralFWtaZjpeiWmax4mA53udd6F5WNCiHTcI+NUNe
9bekG6qn0u4+aFLrrJ96SroYL6+esjrHsRp1/7J7rWtn+uzydkADtfaWojOBpeRoWTOWrrk/9UyAMrW0GiinCqS9FzQNpiNdFwmG
Kr2+9c54EwDf3BmjgurIuZ9y0+GpO3NnjEK2LaQs1remdUSzIt1VwSlxgr1ccvRQwiA79bOa8PLoV+EVSSpvRRlLVqXFcVBz1bfq
68mZvqGwblZ9lnIetz7phrEO7V6ngVeeMuz5nE4pF9q+m8zoln4TYJHtqY+nw70nudZJFs1vgVsZKUt0kWCd1SNppy6YdCzX+QBL
nWocL2pSckGPrmllruKI8bo5jvinBrtwMDfzFPgNzhRkMtBfO/HsfKahnyEuvzKl0X68zhjdj8uqTK/6QUY6497r/mkWnn2B5Np3
QEz1srtYiM4Uv++6ft8gKeWQXyqXe3fQN2/20jszvKpcG5+OYadzWGcjSg6mE48hTz1Yn7H6NQFModouE2btFOZQDhEl+mt1a4N6
CAm3bjmElFgQeZKFTveJARxgMebCun3Tp8YuM+VvhFBJpr9cflTLQ+V71YStcZmtDAIDtfl35k+au1Z9/zjs0lpYDiVCE33Krhlx
ShZDMsLODLW+DAct0iXGW6FzYYL0p89Fn2cq8bhykB7MV0gx2kkF3/d5nJYfAAhNZpyMXpF2KjgrjLq+9p2seeHFOReQrN3rnAWp
V97pHFeNFe1hnVioF2eWMx9Tqy3ESen9M/eHySktbSvQKQA7ubQfXQ1qbq5GnNz05zgqwbVBQAY/kAbSDKUHfAp1EnovpFmvyL83
upDf7VqttZdIDjPs1t8eKpUG4YhcAQWletlJ8yJ7yv0irzG+pIxVd6wO7JpP4/ZvEHN06wV6YL3a2NR3OsRk3JdEqgJz9ezZy4dr
YNC0GGYFtOkpCOfqTqwvLsqQ2unmwWhcTLao9NrXt2VGYx1wLmHZzF/PNaK52dKgsYg4vA6UgWM/3VKI+uffM0NEXjDPpMqx5qWU
3Nhcz/uZIl3DNttMxihPhe10Og2JyQ1kHxcL/kJIgJxTSPFNdAzvHXrrCVglAVSt5B7QsPeP8WFCM2tlETGMNZ5zwRF/SUagfy2h
70Bpq7vR7DdwXptH2g/YQMa7vCTsp+SQ2ThzWa6o+pyYSZtXMpcQrNa3kFUKM4fD9limU2t7e/SQJq78Wowjng0tKnpfD5h4q+XZ
P820mfOVIWhPqYsXDQMvtFXvwZ+LRxFOFbMkvTsdsRDxdFxolBWxhKAYz+UUABtQQzw3Eln7D5oMGe5vrrQAADX3rp82O2xOFXuV
UOeEplc4z5gRyli3bBBzyLmxeG9tA6aK5qd1BotfxrVaSBB3Cp8OItAHduXHF1ZAjav+aM4H27bF9F+eXar1rd0G642VIwi0fzpT
tLMu86YE67YBZW3n5bpKarypq7Zg+jb6eY/le394nj0l67wOkrHdr8vHw+OtI9g5y3OPxxrHFMXyIPTGxPISayZmLLP8EHaV5Soj
3GKmB0rfavHCiHcPNImjyaO/wJfZbkyZCUyGzabydgdoBRbWyjPHLpk1eSYQWcxRWNUt5/rLyu7NMLW9Y1boEMCr+ZqqibqxeciH
AY21IGFW3ilvLx7wW89Zl9nLywsAYTo+cWDMhXvL6aEjaiwbyrCRiheoHnKvI9ceD4glOB6/94NkonSNqZ+ZLltPBjrSBEJy9eQS
1rS0IGWGxjJC2JnY50t2dEvWFsorGMZzf9F5C+8ipvlKpTatT/8FvDKP9Vzp2Hj6iTDVTwL+4DAglS0izyFBca+KOdZKDLkc63Me
pmvFJkgMBiYPZJz1ZFjTMcU/8RL3sgI1ElTNdXOfYwOieriz1mhed/qEyKBv8xr2OEqxOiSxyMFVhATVPTSHgwClmZ7nBaIu9yAN
WGTdi73NdxiusLHyWBmu+nFSld7THgAu86GLHKjB8M9KYIVacwPagS/Y1KuMv0RXY77Da2gANbcSod5o06Z9LVWo8Cwt5lHSrhz9
OfAOQvfAyxpH7OpcfbqMQPelmGHIqmA5b4OKWMsxVhvZQyA901lqV5orO2oEfwjh1JX2CWm2VXbwm0AclBrzbmmnj/MeCocW83zb
6S8jYGGRKyeZhVr62qCFnpIr+ZptfrrcTMjS0CIXDYqeGH+wMUHTH+3h9acfY2YZzKJxwXB9nAiNBdBedr3GsZ9P/dEQBBTeY2NH
kENtp1cgQ2bj4znxupvT5G9QzmrkEZif9sM72FnAprEqD+PUs4xDQxrG3mJZq5CAvcXFcQnARLJ2J933kx10rJvKdPk4rkzseLo4
MYl5CUvau7H7k/eRWFLOqCvyg9sZBopZsRxHRrG3SduP+QCiYbFaaMA5PsCJN/W7I8mySGRD8yzMvX7ENoleogFx62zmRmZNlw7F
f7u/8dyUUm+uAeDbHUI9boiZX9QiCAII8ATqILZotcXabFaQHi+F0Nt5rZf+jfZ0C2KCw4K/III65AcfNfPc/IVvGYuU/Nr0Bcz9
1bbgFr3U8RxPASVrbCxQCjHqy1idWypRKTA+13axnTNGp9CXC3hXBkN+ONuYCqGrIopBFwvePKZyrX6gZL7jZa6arsqmyiYDr5PL
WOS5mpzSS1pemdwYUvoqAOmAfYUue1x8xcDpOMeOU4Z9cK8WJpjHvFj06AiG6SSiAED6E3Z1qpr4IX+1oyxwtMerhXfhiq5aasg3
0XJjpTHbY8ViZqCQ5BE+7+81lEgrUiewAqHXo7zZEtGt/7oK2kU9i8O7qkLzlbsKiT3+tF3p+HMc7mDOA/TEUnv/+2iLDZhzqef9
rCDJsXoJZkzNp3MotnCVfnWmoFV/nlkcF63DzkTW8dh30Bbq1XqrCwIWfzDfma7BFll9HFI+fUkpZVcfmwTW81C9nJ/J0Dg9GFnb
z36pJe++KMKijCs1ZO62zWzrME9ma623ddggHearIyYTCLze9qwrfU6SMev902PH4Vi+KlhC5rw9hc7kY5ZoWuzboS2fnMw9zBgG
6bErh7HfydubbtAPfJkdg7nR+iLvmbYk1/bAcNk7AHikEZIw1RzaEDFII8Vjr7APsvSvsU6tHo9PrADqfXdWrdv08lQf6D+TfrX4
xL0Mn1h7zS22h6QZmnJ+MuJmVDV/HULpZ3nyAnC0LNqIrwG+Fi+2Kz4PDlO/hp1sKPHjueV2Nf2jE/CYwCGqDQjaYwPQOrpXs4iH
x2fb7dh6jq0LPAWzi4+rJlfbLKJbTebPmV4x+01RO3uCB5ECeGH9oBXPOPQUOuo3NQmdkfJ0eNenyxhL05NobGylnFme3C6K7WSx
hxuECOS5PfcOomvt6rJIx4aXIXkak9BgybH/YGZmoKce0iXaemxIVRi4qU5AjV0/Utjf89zobbv6nmEZp6LBQ/644ny1LVO6De3Z
PtNrc19tzegHlmc7ifWqJXb9K+yqI/JkDrJjpBfFrG2xctAzQffpsP1TimSt0MHNu8H5Ldtpo2ZsimxPdgVIdboV2H6VbjZ9BABZ
9Pjy5rjOsnSnT0i0XOI1sV2XSrimketqxknvpJ6DJV3GNoeVKRt27kdMvx6h6oXG4bypL76mqaC87uQ183ZXz0d2piw/6JnVNRrf
y6qq7R72pD4GgFr/unsyYy6WTHDfmZAu3BP7JXeTu7veLAfWsdrTFjupHUKzOWTyzva9Q24rZabnzC4xcbkMObvoPuOwi6sTci0M
a7cnUSuz0UbKd7vfUzvpuSkA7sxZWBOxt6ZPQDHVvwbA7GyQHopi6Mk7Eaz2haPl7BePf2O/jNXZl6lU7T4y6MhTxrB/z2LOmzFp
1VpZlDuLte3JrQXOaGV8jYVxAP1t4Aj+69a+DsXOqKCEstotgyHkJO2d5cG89J+N5OmVJ0GMTWVq0ABMRxxnXYydfgub+POcLNjq
2zElVbuik3MAAebacmV5qE/JTkcatbQyJ9ybiI4zDvT0TcAYvKf5pZmMux5Utl5+m1UmRmmiNFW+L/ebK5hOPK6mzFS9p5O3Zwuy
jrIG5oHByJydvz5NbefjEn4XZbalikpZZ1f9+jBHKSC6UJ86d1ny02yJ42Y/3TgOxKye8Dolx7pRJqXvv7Ihw8XKSteqp4PRc5VD
uJw/pw7153SXlBEJhq1+3W3op8JufB9pdoiAXp+kT7bNK1+faLb+kvykl7Ivo6fTLN4GSjsNpz2tNbF0OXZxp/NYu91KiAqoStwY
qyqbu2f92GBCfG39K73brsNL50kt6chQ7A9yqvR5x66MMOyKuPtdD3WygfvdD7h6TFGP8GNDwsiaxrJUq6/Sa3o6Dn49c8lQL4YV
tvmw1eMVyrx7KpkPmNGeq5cvi1n3RK4MjOmCsW+tMa/QzZSjTZh9kOO6WEDgURfX2Zr7p1IhJfPlsZupoJw6tkaFcU9h96yKHLTd
ieuSOmHwXZ/LVbHnR1zVDH04v7nG7N30aigtQGietE0HUE1RiDGHoLm75iDxLtKuRsUUJr4sH9fZ1jq24jd2c/CEWOBQUG/8Ygbz
Qbo7rQ+w7dmyXD2pC7On9/PC2rf46QBcHSSc7wsCarR873scoguWDCNE13ogy3wasEqtNynRsMgv0GRRXr/6M7ObR9E3a5be6NBr
Zc0XvCn8XkVfE+C02/lzPwmXq1VzT12eD0+w2WzJX/dj6V5I5t7XwZ6f42q93ZWh5scXw1BwvTpDz1L3/IQFYfAVWc2YU5sd/x8z
2xhCiRoHaBeAX06KalbL4RJxY2wDeVJTS2896D+ihJPZM1gYrfGrB202b5TH18suBmycFtdhWrz61M64XFLsW9uYh+Y1Nsfno6Y9
tu/FddSc5c3g1UymiczAZpm1PlF1pk3mpNd569HV7bhZmBb+3e+shfYHSd1LUr6DHl4Lc3xxTBBcOL0xCzz4h9jgQ9cRV/z0CfBC
tDANcD1n3y17s4PZrTLHrwCA+odnvfkOiGgWfUN5FG8gfRxAymKf2EucNTDHgzJbtay/0z7RHydUa/vmB4M0b7nkJAVZ+OFAFm15
4rP7R8e0HeJGyArtRxULfoH/zROb3novh3JWz3ZSietqac4Icb6qrdP8lo57PtY4YQafveoxMW1t9U5PvXtdjUtMNmKuEb4xxK3u
s4FGB8V8C4HES/pi7B+92U9P+cGYx+dPK2PDKjnezPmlmPtLNmD+4CQuzjDLT8jslh25PbfDOveav761vWj5QWb84E6/2gMDoJ0P
/Py1dcatsKvm5zxknPtNDmDJ4NdP6dlFyi2Ge+kOlFmX87jEEzsLxBACO2zZaK9T9q+vJCx3zJjR1QFd9Lg6TlkA1OXUULnX4zwJ
vkd+vC5y1sR9jqCdHNkk95vHmreBOI+N3WMuRxJEpLwOHJo2y0HWaCPZj9VKlhIVGt2lp5/h4UaV1C9/GT2f3iCuHK7tliMozlPW
ueBjjumeZ66r8JsT6ehdHWlcfjRW33lW11n237iM86kDfiUgLX4t3kmXHpjGRN6AKAqkgvdL66fqBdxr8aCZtX9edeYz8RLhNU4h
1O4msP/BasJCIVdKj8CUwMoDev4idgq9zAIlmbkPwxGg3L1HaZgxebWG8pd8fQtN2VFznJ//56sJH7qwfLKyl3qlO9BrIz2iRqFw
4Xx2IhzvV7SgwcU7UC8tzPLwbu/3hSq/xbCha/I0HTvpzP/5ssIO5kegLpQ7Qxfp/agNs/CrXQBhhE/wDJlib6ZrsmTtLdQYbJue
L7zCNvThs2e6VJmR30rsv7+UMkyNtTz2JtvuthPegd0s39HSqNp42o5cZ3glAjWzdKSGBx3pyx7L1wL420zeICG21iUa/ZBGzdM5
w2f5Wr3xaydQzE/OByzCnO8bbwQT5Yn1MvlE6ve6GPSq+ftr6adtgJ4glLS6DMZGWK8nqkaAdZ0GrqDUN4RZGHWwgAzMpoL/EaVj
OcFlYLN5sDcjPuEs9pa5DV7cu7yfZRtETuX6FFRhhYZG99QcB4vMLquksEKjvrfP1Ei9rA5mKHguN1vY55u6Mz8r1B/sSJlh6frm
EXu5tPc4+qw6i6/jf0p9uII6KLXPqGPg5jxQfoBlf9wnQDpGslO9Z67VJR/bR5QWvzFEHep5hkc58gNa1wd1mPV0VBVbl4TbZBLa
+BlXhzLI15dm6L208SSUz87S2q+vh/Fzef46oxftNrTxdi1vdR/D4qbXV5uquSP6LJ8fVumXrKHDSt8iM9ySaI04m3n0xZ6kRibP
eNeAj3YgGKo+IopBmPV1MSOMOJiNtexy+T/4Pd7yfgOLaQwyItDgRxpOp+cR7JzcvnMC6A9gbvp+IxOChD2Y6giVDrO6tYt6Avj5
RCaThL7nROk/ixWVMCBeO7srhvwOduqP3lQj4ji1W8CZ7fsAls22VXdiWacnsV5EbLSfz2fcWJ/9MQdR1ZOlyjrz0uQSWlL8668S
qu9gqK95emavulfn6UzNiOxO5mtyqgqZbrKsdBEJ1URMJQkEyK5SKj+qmP6m7g16mqnry3HAzxrYeD+RBfIZF8SAEjEvpgimRQM4
WtYn/QXyfCt4+m9qCUTEaoTW3oJB2hdFri9uSTudlU4BLOj3chXjZ/20yDiFgzyHcrsPiLvfpDzW4KT4eVnmLXf9kWlG/8j6iBYz
qL1p1kFIQOulfV/xwtm9RRqsx+gtcj2xeCk/vtH4V+8F//zlL3//l/8P+OX6lQ==
"""
boundary_bytes=zlib.decompress(base64.b64decode(BOUNDARY_COMPRESSED))
assert hashlib.sha256(boundary_bytes).hexdigest()==BOUNDARY_SHA256
boundary=json.loads(boundary_bytes)
REGION=shape(boundary['features'][0]['geometry'])
assert REGION.is_valid
REGION_WKT=REGION.wkt
(RUN/'region.geojson').write_bytes(boundary_bytes)
print('Study boundary verified:',REGION.bounds)


Study boundary verified: (79.65085150000002, 5.919017299999974, 81.87896019999998, 9.8355856)


## 4. Record the source object and inspect the actual schema

An ETag is an object fingerprint, not a scientific checksum. It is recorded with length and Last-Modified and checked again after the queries. The regional extract gets its own SHA256. This does not pin the entire global catalogue or independently verify the NASA observation record.

Only the exact `detections.parquet` is read. Do not glob other files in the year directory: aggregates and rolling files have different meanings and can duplicate observations.


In [16]:
def source_fingerprint():
    response=requests.head(SOURCE,allow_redirects=True,timeout=60)
    response.raise_for_status()
    result={key:response.headers.get(key) for key in ['ETag','Content-Length','Last-Modified']}
    assert result['ETag'] and result['Content-Length'], 'Source fingerprint unavailable; review before extracting.'
    return result

BEFORE=source_fingerprint()
con=duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute('SET threads=2')
con.execute('INSTALL httpfs');con.execute('LOAD httpfs')
con.execute('INSTALL spatial');con.execute('LOAD spatial')
SCHEMA=con.execute('DESCRIBE SELECT * FROM read_parquet(?)',[SOURCE]).df()
display(SCHEMA)
SCHEMA.to_csv(RUN/'source_schema.csv',index=False)
required={'geometry','acq_date','acq_datetime','sensor','satellite','quality','confidence','confidence_pct','type','frp','daynight'}
assert required<=set(SCHEMA.column_name), 'Source schema changed; review it before proceeding.'
geometry_type=str(SCHEMA.set_index('column_name').loc['geometry','column_type'])
assert geometry_type.startswith('GEOMETRY'), 'Geometry was not decoded. Check DuckDB/spatial versions before proceeding.'
assert not {'geometry_wkb','audit_lon','audit_lat'}&set(SCHEMA.column_name), 'Audit column-name collision.'
print('Source schema verified. Original sensor/confidence/type fields will be retained.')


,column_name,column_type,null,key,default,extra
0,acq_datetime,TIMESTAMP,YES,None,None,None
1,acq_date,DATE,YES,None,None,None
2,sensor,VARCHAR,YES,None,None,None
3,satellite,VARCHAR,YES,None,None,None
4,instrument,VARCHAR,YES,None,None,None
5,quality,VARCHAR,YES,None,None,None
6,version,VARCHAR,YES,None,None,None
7,brightness,DOUBLE,YES,None,None,None
8,bright_t31,DOUBLE,YES,None,None,None
9,bright_ti4,DOUBLE,YES,None,None,None


Source schema verified. Original sensor/confidence/type fields will be retained.


## 5. Extract the region and independently reconcile its row count

A bounding box narrows the remote query, then the polygon selects detection centres. All published satellites, confidence values and source types are retained. Unknown source type remains unknown.

The output is ordinary Parquet with original attributes, original geometry encoded as WKB, and audit longitude/latitude. Its coordinate reference is recorded as CRS84. Query success means successful retrieval of matching mirror records, not verified observing coverage.


In [17]:
start=f'{YEAR}-01-01';end=f'{YEAR+1}-01-01'
WHERE_SQL="""
acq_date >= CAST(? AS DATE) AND acq_date < CAST(? AS DATE)
AND ST_Intersects(geometry,ST_MakeEnvelope(?,?,?,?))
AND ST_Covers(ST_GeomFromText(?),geometry)
"""
PARAMS=[SOURCE,start,end,*BBOX,REGION_WKT]
EXTRACT_SQL=f"""
SELECT * EXCLUDE (geometry),
       ST_AsWKB(geometry) AS geometry_wkb,
       ST_X(geometry) AS audit_lon, ST_Y(geometry) AS audit_lat
FROM read_parquet(?)
WHERE {WHERE_SQL}
"""
OBS=con.execute(EXTRACT_SQL,PARAMS).df()
assert len(OBS)>0, 'The positive control year returned no observations; inspect the source/query.'
DIRECT_COUNT=int(con.execute(f'SELECT count(*) FROM read_parquet(?) WHERE {WHERE_SQL}',PARAMS).fetchone()[0])
assert len(OBS)==DIRECT_COUNT, 'Remote count and extracted rows differ.'
AFTER=source_fingerprint()
assert BEFORE==AFTER, 'Source changed during extraction; create a fresh run.'
OBS['geometry_wkb']=OBS.geometry_wkb.map(bytes)
assert OBS.audit_lon.notna().all() and OBS.audit_lat.notna().all()
assert pd.to_datetime(OBS.acq_date).dt.year.eq(YEAR).all()
assert pd.to_datetime(OBS.acq_datetime).dt.date.eq(pd.to_datetime(OBS.acq_date).dt.date).all()
OBS.to_parquet(WORK/'observations.parquet',index=False)
print('Verified extracted rows:',len(OBS))
display(OBS.groupby(['satellite','sensor','quality'],dropna=False).size().rename('detections').reset_index())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified extracted rows: 13366


,satellite,sensor,quality,detections
0,Aqua,MODIS,sp,642
1,NOAA-20,VIIRS_NOAA20,sp,6094
2,Suomi-NPP,VIIRS_SNPP,sp,6432
3,Terra,MODIS,sp,198


## 6. Make observed summaries without cross-sensor calibration

These are detection counts per satellite, not unique fires, burned area, or harmonized activity. Zero means no matching detections in the successfully queried mirror, with observing coverage unverified. Do not replace a failed query with an empty DataFrame.

The pilot uses the four platforms documented for 2023. A newly appearing platform triggers review. No MODIS/VIIRS confidence crosswalk or inferred source category is applied.


In [18]:
EXPECTED_SATELLITES=['Terra','Aqua','Suomi-NPP','NOAA-20']
assert set(OBS.satellite)<=set(EXPECTED_SATELLITES), 'Unexpected platform: inspect source availability and update the explicit contract.'
OBS['audit_month']=pd.to_datetime(OBS.acq_date).dt.strftime('%Y-%m')
counts=OBS.groupby(['audit_month','satellite']).size()
index=pd.MultiIndex.from_product([[f'{YEAR}-{m:02d}' for m in range(1,13)],EXPECTED_SATELLITES],names=['month','satellite'])
MONTHLY=counts.reindex(index,fill_value=0).rename('recorded_detections').reset_index()
MONTHLY['extraction_status']='query_completed'
MONTHLY['observing_coverage']='unverified'
assert int(MONTHLY.recorded_detections.sum())==len(OBS)
assert not MONTHLY.duplicated(['month','satellite']).any()
MONTHLY.to_csv(WORK/'monthly_observed_counts.csv',index=False)
SENSOR_COUNTS=OBS.groupby(['satellite','sensor','quality'],dropna=False).size().rename('recorded_detections').reset_index()
SENSOR_COUNTS.to_csv(WORK/'sensor_counts.csv',index=False)
DUPLICATE_ROWS=int(OBS.duplicated().sum())
print('Exact duplicate extract rows:',DUPLICATE_ROWS)
if DUPLICATE_ROWS:print('Review source duplicates before using this extract in the app; no rows were silently dropped.')
display(MONTHLY.pivot(index='month',columns='satellite',values='recorded_detections'))
display(OBS.groupby(['quality','type'],dropna=False).size().rename('recorded_detections').reset_index())


Exact duplicate extract rows: 0


satellite,Aqua,NOAA-20,Suomi-NPP,Terra
month,,,,
2023-01,3,67,43,2
2023-02,13,168,136,3
2023-03,39,270,206,9
2023-04,7,192,284,11
2023-05,1,71,91,4
2023-06,46,270,289,8
2023-07,151,959,1223,39
2023-08,322,3168,3424,103
2023-09,47,838,666,15


,quality,type,recorded_detections
0,sp,0,13264
1,sp,2,77
2,sp,3,25


## 7. Save a checked handoff bundle

The source fingerprint, schema, SQL, geometry hash, extract checksum and row reconciliation travel with the observations. Original extracted attributes are preserved in the Parquet file; monthly CSVs contain no harmonization.

Download the ZIP from Google Drive if you want to hand the actual pilot data back for the explorer implementation. Saving this executed notebook back to the project also lets us review the checks. No browser-specific download helper is required.


In [19]:
import shutil,zipfile
(WORK/'source_schema.csv').write_bytes((RUN/'source_schema.csv').read_bytes())
(WORK/'region.geojson').write_bytes(boundary_bytes)
manifest={'run_id':RUN_ID,'year':YEAR,'region':'lk','unit':'recorded detection observations per satellite',
    'source_url':SOURCE,'source_before':BEFORE,'source_after':AFTER,
    'source_etag_is_not_a_checksum':True,'boundary_sha256':BOUNDARY_SHA256,'coordinate_reference':'OGC:CRS84',
    'extract_sql':EXTRACT_SQL,'query_parameters':PARAMS,'extracted_rows':len(OBS),'direct_source_count':DIRECT_COUNT,
    'monthly_count_sum':int(MONTHLY.recorded_detections.sum()),'exact_duplicate_rows':DUPLICATE_ROWS,
    'packages':versions,'harmonization_applied':False,'observing_coverage_verified':False,
    'production_export_enabled':False,'files':{}}
for filename in ['observations.parquet','monthly_observed_counts.csv','sensor_counts.csv','source_schema.csv','region.geojson']:
    path=WORK/filename
    manifest['files'][filename]={'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
(WORK/'audit_manifest.json').write_text(json.dumps(manifest,indent=2,default=str,allow_nan=False))
for path in WORK.iterdir():
    if path.is_file():shutil.copy2(path,RUN/path.name)
ZIP=RUN/'cindertrace_atlas_lk_2023_pilot.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(WORK.iterdir()):
        if path.is_file():archive.write(path,arcname=path.name)
print('RUN_ID:',RUN_ID)
print('Rows:',len(OBS),'| independent source count:',DIRECT_COUNT,'| monthly sum:',int(MONTHLY.recorded_detections.sum()))
print('Duplicate rows requiring review:',DUPLICATE_ROWS)
print('BUNDLE:',ZIP)
print('Bundle SHA256:',hashlib.sha256(ZIP.read_bytes()).hexdigest())
print('Observed-data pilot saved. Share the executed notebook and, for app integration, the ZIP bundle.')


RUN_ID: 20261114T183601Z_07884c3d
Rows: 13366 | independent source count: 13366 | monthly sum: 13366
Duplicate rows requiring review: 0
BUNDLE: /content/drive/MyDrive/cindertrace_atlas/runs/20261114T183601Z_07884c3d/cindertrace_atlas_lk_2023_pilot.zip
Bundle SHA256: 11184caea4d9ca669e4f7f455d120d8f4d898db0bf2086f5c86cb9b47650e49e
Observed-data pilot saved. Share the executed notebook and, for app integration, the ZIP bundle.


## Checkpoint

Review the actual source schema, satellite/quality breakdown, monthly table, duplicate count, and three matching row totals. A nonzero duplicate count or changed fingerprint blocks app integration until investigated.

Next milestone: use this checked observed-data bundle to build the map/time-brush/calendar interaction. Expand data scope after source and query checks pass. Independent harmonization evaluation follows as a separate layer; none of the old estimates or validation scores are imported.
